# GPUs & your first Colab notebook

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/venkathub/larnix/blob/main/modules/05-deep-learning/ch11-gpus-first-colab-colab.ipynb)

**Colab companion of `ch11-gpus-first-colab.qmd`.** Generated by `infra/ci/make_colab.py` (P2-D8); do not edit by hand — edit the chapter and regenerate. Run cells top to bottom; exercises grade themselves with `run_tests`. Free tier is enough: Runtime → Change runtime type → **T4 GPU** (₹0).

In [ ]:
# Torch version-floor guard (P2-D11): Colab preinstalls torch and rolls it
# forward, so we assert a floor rather than pin an exact version.
import torch

def _ver(v):
    return tuple(int("".join(ch for ch in p if ch.isdigit()) or 0)
                 for p in v.split("+")[0].split(".")[:3])

_FLOOR = "2.2"
assert _ver(torch.__version__) >= _ver(_FLOOR), (
    f"This notebook needs torch >= {_FLOOR}, but this runtime has "
    f"{torch.__version__}. In Colab try Runtime -> Disconnect and delete runtime, then reconnect; if it persists, please open an issue."
)
print(f"torch {torch.__version__} (floor {_FLOOR}) OK")

In [ ]:
import os

# CI runs this notebook CPU-scaled (LARNIX_CI=1): smaller matrices, fewer
# repeats. On Colab you get the full sizes.
LARNIX_CI = os.environ.get("LARNIX_CI") == "1"
SIZE = 256 if LARNIX_CI else 2048
REPEATS = 3 if LARNIX_CI else 10
print(f"matmul size {SIZE}x{SIZE}, {REPEATS} repeats"
      + ("  [CI-scaled]" if LARNIX_CI else ""))

In [ ]:
# Larnix grader bootstrap (single-sourced — P1-D9/P2-D8). Colab has no
# quarto-live VFS, so fetch lib/grader.py from the repo (Colab is online);
# offline, in CI, or when the fetched copy can't serve the imports the
# companion needs (version skew — found live 2026-07-28), use the inlined
# copy embedded at generation time. Capability is validated, not fetch
# success: a stale-but-fetchable grader must not win over the fallback.
_LARNIX_GRADER_FALLBACK = r'''"""Larnix in-browser assert-grader helper.

Runs both inside a `{pyodide}` cell (in the learner's browser) and in CPython
(for these unit tests), so the grading logic is verifiable off-browser.

It gives every auto-graded exercise one consistent pass/fail UX:

    from grader import run_tests        # or paste this file into a `setup` cell
    run_tests([
        ("all correct", accuracy([1, 1], [1, 1]), 1.0),
        ("half right",  accuracy([1, 0], [1, 1]), 0.5),
    ])

Loading it in a chapter (single source of truth — P1-D9 / DECISIONS D0016):
add this file to the page's `resources:` front-matter key so quarto-live copies
it into the Pyodide VFS at startup, then `from lib.grader import run_tests` in a
hidden `#| edit: false` cell. No more pasting the helper into every chapter.

On success it prints "All N tests passed ✅". On the first set of failures it
prints each result and raises AssertionError, so the cell shows an error.

Two grading modes (P2-D7, DECISIONS D0020):

1. **Seeded-deterministic** (browser / CPU-twin chapters): the exercise pins
   every seed, so plain ``(label, got, expected)`` tuples with the existing
   float tolerance are enough. Nothing new to import.
2. **Property-based** (colab / GPU chapters, where training is stochastic):
   assert *properties of a successful run* instead of exact values, via the
   check-builders below, which drop straight into ``run_tests``:

    run_tests([
        between("test accuracy", acc, 0.95, 1.0),
        decreased("training loss", first_loss, final_loss, min_drop=0.80),
        changed("weights updated", w_before, w_after),
        grad_check("dL/dw", loss_fn, w, analytic_grad),
    ])

Design notes:
- Pyodide-safe: standard library only (no imports beyond the stdlib).
- Floats compare with a tolerance (default 1e-9) so exercises that compute
  e.g. an accuracy don't fail on representation error.
- Check-builders evaluate immediately and return a ``_Check`` record;
  ``run_tests`` prints them with the same ✅/❌ UX as classic tuples.
"""
from __future__ import annotations

_NUMERIC = (int, float)


def _is_number(x) -> bool:
    # bool is a subclass of int; treat True/False as non-numeric for tolerance.
    return isinstance(x, _NUMERIC) and not isinstance(x, bool)


def _equalish(got, expected, tol: float) -> bool:
    if _is_number(got) and _is_number(expected):
        return abs(got - expected) <= tol
    return got == expected


class _Check:
    """A pre-evaluated property check (P2-D7 property-based grading mode).

    Built by :func:`between`, :func:`decreased`, :func:`changed` and
    :func:`grad_check`; consumed by :func:`run_tests`. ``got`` and
    ``expected`` are pre-formatted, learner-readable strings.
    """

    __slots__ = ("label", "ok", "got", "expected")

    def __init__(self, label: str, ok: bool, got: str, expected: str):
        self.label = label
        self.ok = bool(ok)
        self.got = got
        self.expected = expected


def between(label: str, value, low, high) -> _Check:
    """Check ``low <= value <= high`` (bounds inclusive).

    The property-mode workhorse for training exercises, e.g.
    ``between("test accuracy", acc, 0.95, 1.0)``.
    """
    ok = _is_number(value) and low <= value <= high
    return _Check(label, ok, repr(value), f"a number between {low!r} and {high!r}")


def decreased(label: str, before, after, min_drop: float = 0.0) -> _Check:
    """Check that ``after`` fell below ``before``.

    With ``min_drop`` (a fraction, e.g. ``0.8`` = "fell by at least 80%"),
    also require ``before - after >= min_drop * abs(before)`` — the
    "loss fell ≥ N% during training" property assert.
    """
    numeric = _is_number(before) and _is_number(after)
    if numeric and min_drop > 0:
        ok = after < before and (before - after) >= min_drop * abs(before)
        expected = f"a drop of at least {min_drop * 100:g}% from {before!r}"
    else:
        ok = numeric and after < before
        expected = f"any decrease from {before!r}"
    return _Check(label, ok, f"{before!r} -> {after!r}", expected)


def _differs(a, b, tol: float) -> bool:
    """True if a and b differ by more than tol (recursing into sequences)."""
    if _is_number(a) and _is_number(b):
        return abs(a - b) > tol
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return True
        return any(_differs(x, y, tol) for x, y in zip(a, b))
    return a != b


def changed(label: str, before, after, tol: float = 1e-12) -> _Check:
    """Check that a value (or nested list/tuple of values) actually changed.

    The "weights actually moved after one training step" property assert.
    """
    ok = _differs(before, after, tol)
    return _Check(
        label,
        ok,
        "values changed" if ok else "values unchanged",
        f"a change larger than tol={tol:g}",
    )


def grad_check(
    label: str, f, x, analytic, eps: float = 1e-5, tol: float = 1e-4
) -> _Check:
    """Check an analytic gradient of scalar ``f`` against central finite differences.

    ``x`` and ``analytic`` are either both scalars, or a flat list/tuple of
    parameter values and the claimed gradient per parameter. For each
    coordinate i the numeric gradient ``(f(x + eps*e_i) - f(x - eps*e_i)) /
    (2*eps)`` is compared to ``analytic[i]`` by relative error
    ``|num - ana| / max(1, |num|, |ana|)``; the check passes if the largest
    relative error is <= ``tol``.
    """
    if _is_number(x):
        xs, ans = [x], [analytic]
        call = lambda vals: f(vals[0])  # noqa: E731
    else:
        xs, ans = list(x), list(analytic)
        call = lambda vals: f(list(vals))  # noqa: E731
    if len(xs) != len(ans):
        return _Check(
            label,
            False,
            f"{len(ans)} gradient value(s) for {len(xs)} parameter(s)",
            "one gradient value per parameter",
        )
    max_err = 0.0
    for i in range(len(xs)):
        plus = list(xs)
        minus = list(xs)
        plus[i] += eps
        minus[i] -= eps
        num = (call(plus) - call(minus)) / (2 * eps)
        ana = ans[i]
        if not _is_number(ana):
            return _Check(label, False, f"non-numeric gradient {ana!r}", "a number")
        err = abs(num - ana) / max(1.0, abs(num), abs(ana))
        max_err = max(max_err, err)
    ok = max_err <= tol
    return _Check(
        label,
        ok,
        f"max relative gradient error {max_err:.2e}",
        f"<= {tol:g} (analytic vs finite-difference)",
    )


def run_tests(tests, tol: float = 1e-9) -> bool:
    """Run a list of checks: ``(label, got, expected)`` tuples and/or
    property checks built by :func:`between` / :func:`decreased` /
    :func:`changed` / :func:`grad_check` — freely mixed.

    Prints one line per check. Returns True if all pass; otherwise raises
    AssertionError after printing every result.
    """
    tests = list(tests)
    failures = []
    for item in tests:
        if isinstance(item, _Check):
            label, ok = item.label, item.ok
            line = f"{'✅' if ok else '❌'} {label}: got {item.got}"
            if not ok:
                line += f", expected {item.expected}"
        else:
            label, got, expected = item
            ok = _equalish(got, expected, tol)
            line = f"{'✅' if ok else '❌'} {label}: got {got!r}"
            if not ok:
                line += f", expected {expected!r}"
        print(line)
        if not ok:
            failures.append(label)

    if failures:
        raise AssertionError(
            f"{len(failures)} of {len(tests)} test(s) failed: " + ", ".join(failures)
        )
    print(f"\nAll {len(tests)} tests passed ✅")
    return True
'''

import importlib, os, pathlib, sys
pathlib.Path("lib").mkdir(exist_ok=True)
_NEEDED = ('run_tests', 'between', 'decreased', 'changed', 'grad_check')

def _grader_serves():
    try:
        sys.modules.pop('lib.grader', None)
        sys.modules.pop('lib', None)
        _g = importlib.import_module('lib.grader')
        return all(hasattr(_g, _n) for _n in _NEEDED)
    except Exception:
        return False

_fetched = False
if not os.environ.get("LARNIX_CI"):
    try:
        import urllib.request
        with urllib.request.urlopen("https://raw.githubusercontent.com/venkathub/larnix/main/lib/grader.py", timeout=10) as _r:
            pathlib.Path("lib/grader.py").write_bytes(_r.read())
        _fetched = _grader_serves()
    except Exception:
        pass
if not _fetched:
    pathlib.Path("lib/grader.py").write_text(_LARNIX_GRADER_FALLBACK)
    assert _grader_serves(), 'inlined grader failed to import (please open an issue)'
from lib.grader import run_tests, between, decreased, changed, grad_check
print("Larnix grader ready" + (" (fetched)" if _fetched else " (inlined copy)"))

### Hello, hardware — ask, don't assume

Selecting "T4 GPU" in the runtime menu is a *request*, not a
guarantee — quota limits, reconnects, or a fresh runtime can leave
you CPU-only. So the first working cell asks `torch` what is
actually attached, and tells you how to fix it if the answer is
"nothing":

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}   device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU-only runtime. Runtime -> Change runtime type -> T4 GPU, "
          "then Runtime -> Run all again.")

### The CPU baseline

A big matmul on this VM's CPU. Note the **warm-up call before
timing** — the first run of anything pays one-time setup costs
(memory allocation, code paths warming), and an honest measurement
excludes them:

In [ ]:
import time

a = torch.randn(SIZE, SIZE)
b = torch.randn(SIZE, SIZE)
_ = a @ b                                   # warm-up (untimed)

t0 = time.perf_counter()
for _ in range(REPEATS):
    c = a @ b
cpu_ms = (time.perf_counter() - t0) / REPEATS * 1000
print(f"CPU : {SIZE}x{SIZE} matmul ~ {cpu_ms:.2f} ms")

### The GPU, honestly timed

The moment this chapter exists for — with one GPU subtlety taught
before it bites: GPU calls are **asynchronous**. Python gets control
back before the GPU finishes, so honest timing must call
`torch.cuda.synchronize()` (wait until the queued work is truly
done) before reading the clock. Skip it and you'd clock how fast
Python *queued* the work — flattering and wrong:

In [ ]:
if torch.cuda.is_available():
    ag, bg = a.to("cuda"), b.to("cuda")     # copy the data to GPU memory
    _ = ag @ bg                             # warm-up
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(REPEATS):
        cg = ag @ bg
    torch.cuda.synchronize()                # wait until the GPU is truly done
    gpu_ms = (time.perf_counter() - t0) / REPEATS * 1000

    print(f"GPU : same matmul ~ {gpu_ms:.2f} ms   "
          f"({cpu_ms / gpu_ms:.0f}x faster than this VM's CPU)")
else:
    print("No GPU attached - this cell is waiting for lever 2.")

Two honesty notes on whatever speedup you measure. First, the
comparison is against *Colab's modest VM CPU*, not your laptop —
it's a fair "same machine, two devices" reading, not a universal
constant. Second, **GPUs are not always faster**: `a.to("cuda")`
copies data over a bus, and for small tensors the copy costs more
than the compute saves. Feed a GPU tiny scalar work (your micrograd!)
and it loses to a CPU; feed it big batched tensors and it wins by
orders of magnitude. The GPU didn't repeal Chapter 10's lesson — it
*rewards* it: batch first, then parallelize.

## Practice

Both graded exercises are auto-checked by `run_tests` (set up by the
grader cell earlier) and are written to pass on CPU *or* GPU.
Replace the blanks, run the cell, and the checks tell you how you
did — peek at the solution under each exercise only after an honest
attempt.

### Exercise 1 — Guided (≈4 min) 🟢

Two blanks: pick the device honestly, then move a tensor there.

In [ ]:
device = ____        # "cuda" if a GPU is attached, else "cpu" - ask torch
x = torch.arange(6.0).reshape(2, 3)
x_dev = ____         # move x to that device

run_tests([
    ("a real destination", device in ("cuda", "cpu"), True),
    ("the tensor moved", x_dev.device.type, device),
    ("values survive the move", x_dev.sum().item(), 15.0),
])

<details><summary>Show solution</summary>

```python
device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.arange(6.0).reshape(2, 3)
x_dev = x.to(device)

run_tests([
    ("a real destination", device in ("cuda", "cpu"), True),
    ("the tensor moved", x_dev.device.type, device),
    ("values survive the move", x_dev.sum().item(), 15.0),
])
```

The ask-don't-assume pattern (`is_available()`) is the one you'll
write at the top of every notebook from here on — code that runs
correctly on whatever hardware it lands on.
</details>

### Exercise 2 — Implement (≈8 min) 🟡

Package the chapter's timing discipline into a reusable tool:
`matmul_seconds(n, device)` — mean seconds for one n×n matmul on
`device`, with a warm-up first and `synchronize()` in the right
places. The checks are properties (times are positive and sane; 8×
the work costs more), so they grade fairly on any hardware.

In [ ]:
def matmul_seconds(n, device, repeats=REPEATS):
    """Mean seconds per n x n matmul on `device` (warm up, sync, then time)."""
    ...  # TODO: build a and b on the device, warm up, sync, time the loop, sync

t_small = matmul_seconds(SIZE // 2, "cpu")
t_big = matmul_seconds(SIZE, "cpu")
print(f"CPU {SIZE // 2}: {t_small * 1000:.3f} ms   "
      f"CPU {SIZE}: {t_big * 1000:.3f} ms")
if torch.cuda.is_available():
    t_gpu = matmul_seconds(SIZE, "cuda")
    print(f"GPU {SIZE}: {t_gpu * 1000:.3f} ms   ({t_big / t_gpu:.0f}x)")

run_tests([
    between("smaller matmul time (s)", t_small, 0.0, 30.0),
    between("bigger matmul time (s)", t_big, 0.0, 60.0),
    ("8x the work costs more time", t_big > t_small, True),
])

<details><summary>Show solution</summary>

```python
def matmul_seconds(n, device, repeats=REPEATS):
    """Mean seconds per n x n matmul on `device` (warm up, sync, then time)."""
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    _ = a @ b                                # warm-up (untimed)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        _ = a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeats

t_small = matmul_seconds(SIZE // 2, "cpu")
t_big = matmul_seconds(SIZE, "cpu")
print(f"CPU {SIZE // 2}: {t_small * 1000:.3f} ms   "
      f"CPU {SIZE}: {t_big * 1000:.3f} ms")
if torch.cuda.is_available():
    t_gpu = matmul_seconds(SIZE, "cuda")
    print(f"GPU {SIZE}: {t_gpu * 1000:.3f} ms   ({t_big / t_gpu:.0f}x)")

run_tests([
    between("smaller matmul time (s)", t_small, 0.0, 30.0),
    between("bigger matmul time (s)", t_big, 0.0, 60.0),
    ("8x the work costs more time", t_big > t_small, True),
])
```

Both `synchronize()` calls matter: skip the first and setup drifts
into your timing; skip the second and you clock how fast Python
*queued* the work, not how fast the GPU did it — flattering and
wrong. You'll reuse this tool in Chapter 15's tuning experiments.
</details>

### Exercise 3 — Stretch (optional, ≈15 min) 🔴

Open-ended, **rubric-graded** (not auto-checked). Run this same
notebook on **both providers** — Colab, then Kaggle (File → Download
.ipynb, then kaggle.com → Create → Notebook → File → Import
notebook, and switch the accelerator in Session options). Keep your
first honest run record: fill the template cell once per session,
and keep both records.

In [ ]:
# Your run record - fill in once per session, keep both:
provider = "..."          # "colab" or "kaggle"
gpu_name = "..."          # torch.cuda.get_device_name(0), or "none"
torch_version = torch.__version__
cpu_big_ms = None         # your CPU timing for SIZE x SIZE
gpu_big_ms = None         # your GPU timing (None if no GPU)
print(provider, "|", gpu_name, "|", torch_version, "|",
      cpu_big_ms, "ms CPU |", gpu_big_ms, "ms GPU")

### Done — let go of the GPU

`Runtime → Disconnect and delete runtime`. Free quota is shared;
an idling notebook burns yours. Head back to the chapter page for
the quiz and what's next — Chapter 12 turns this hardware into a
PyTorch training ground.